# Modélisation

Objectif : Créer un modèle pour prédire les maladies cardiaques

Étapes :
1. Préparer les données
2. Standardiser les variables
3. Séparer les données (entraînement/test)
4. Entraîner des modèles
5. Évaluer les performances

In [ ]:
# Importation des bibliothèques
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Évaluation
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import roc_auc_score

# Configuration
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)

print("Bibliothèques importées")

In [ ]:
# Charger les données
df = pd.read_csv('../data/heart_disease_dataset.csv')

print("Données chargées")
print(f"{df.shape[0]} patients, {df.shape[1]} variables")
df.head()

In [ ]:
# Préparation des données
print("Valeurs manquantes par colonne :")
print(df.isnull().sum())

# Remplir les valeurs manquantes
for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype in ['int64', 'float64']:
            df[col].fillna(df[col].median(), inplace=True)
        else:
            df[col].fillna(df[col].mode()[0], inplace=True)

print("Valeurs manquantes après traitement :", df.isnull().sum().sum())

In [ ]:
# Encoder les variables catégorielles
categorical_cols = df.select_dtypes(include=['object']).columns
print(f"Colonnes catégorielles : {list(categorical_cols)}")

label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

print("Encodage terminé")

In [ ]:
# Séparer les variables et la cible
X = df.drop('Heart Disease', axis=1)
y = df['Heart Disease']

print(f"Variables prédictives : {X.shape[1]} variables")
print(f"Variable cible : {y.name}")

In [ ]:
# Séparer les données en entraînement/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Entraînement : {X_train.shape[0]} patients")
print(f"Test : {X_test.shape[0]} patients")
print(f"Proportion malades - Entraînement : {y_train.mean()*100:.1f}%")
print(f"Proportion malades - Test : {y_test.mean()*100:.1f}%")

In [ ]:
# Standardiser les variables
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convertir en DataFrame
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)

print("Standardisation terminée")
print(X_train_scaled.head().round(3))

In [ ]:
# Entraîner les modèles
models = {
    'Régression Logistique': LogisticRegression(random_state=42, max_iter=1000),
    'Forêt Aléatoire': RandomForestClassifier(random_state=42, n_estimators=100),
    'SVM': SVC(random_state=42, probability=True)
}

predictions = {}

for name, model in models.items():
    print(f"Entraînement de {name}...")
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    predictions[name] = y_pred
    accuracy = accuracy_score(y_test, y_pred)
    print(f"{name} - Accuracy: {accuracy:.3f} ({accuracy*100:.1f}%)")

In [ ]:
# Évaluer les performances
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Matrices de Confusion', fontsize=16)

for i, (name, y_pred) in enumerate(predictions.items()):
    print(f"\n{name} :")
    report = classification_report(y_test, y_pred, target_names=['Non Malade', 'Malade'])
    print(report)
    
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['Non Malade', 'Malade'],
                yticklabels=['Non Malade', 'Malade'])
    axes[i].set_title(f'{name}\nAccuracy: {accuracy_score(y_test, y_pred):.3f}')
    axes[i].set_xlabel('Prédiction')
    axes[i].set_ylabel('Réalité')

plt.tight_layout()
plt.show()

In [ ]:
# Comparaison des performances
metrics = []
for name, y_pred in predictions.items():
    accuracy = accuracy_score(y_test, y_pred)
    
    model = models[name]
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
    else:
        y_prob = model.decision_function(X_test_scaled)
    
    auc = roc_auc_score(y_test, y_prob)
    
    metrics.append({
        'Modèle': name,
        'Accuracy': accuracy,
        'AUC': auc
    })

metrics_df = pd.DataFrame(metrics)

# Graphique de comparaison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Accuracy
sns.barplot(data=metrics_df, x='Modèle', y='Accuracy', ax=ax1, palette='viridis')
ax1.set_title('Accuracy des Modèles')
ax1.set_ylim(0, 1)
for i, v in enumerate(metrics_df['Accuracy']):
    ax1.text(i, v + 0.01, f'{v:.3f}', ha='center', va='bottom')

# AUC
sns.barplot(data=metrics_df, x='Modèle', y='AUC', ax=ax2, palette='plasma')
ax2.set_title('AUC des Modèles')
ax2.set_ylim(0, 1)
for i, v in enumerate(metrics_df['AUC']):
    ax2.text(i, v + 0.01, f'{v:.3f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

print("Tableau récapitulatif :")
print(metrics_df.round(3))

In [ ]:
# Trouver le meilleur modèle
print("Meilleur modèle :")

# Basé sur l'accuracy
best_accuracy = metrics_df.loc[metrics_df['Accuracy'].idxmax()]
print(f"Meilleur accuracy : {best_accuracy['Modèle']} ({best_accuracy['Accuracy']:.3f})")

# Basé sur l'AUC
best_auc = metrics_df.loc[metrics_df['AUC'].idxmax()]
print(f"Meilleur AUC : {best_auc['Modèle']} ({best_auc['AUC']:.3f})")

# Meilleur modèle global (moyenne des deux métriques)
metrics_df['Score_Combiné'] = (metrics_df['Accuracy'] + metrics_df['AUC']) / 2
best_overall = metrics_df.loc[metrics_df['Score_Combiné'].idxmax()]
print(f"Meilleur modèle global : {best_overall['Modèle']}")
print(f"Score combiné : {best_overall['Score_Combiné']:.3f}")

In [ ]:
# Tester le meilleur modèle sur un exemple
print("Test du meilleur modèle :")

# Prendre le meilleur modèle
best_model_name = best_overall['Modèle']
best_model = models[best_model_name]

# Prendre un patient de l'ensemble de test comme exemple
example_patient = X_test_scaled.iloc[0:1]
true_label = y_test.iloc[0]

print(f"Patient exemple :")
print(f"Vraie étiquette : {'Malade' if true_label == 1 else 'Non malade'}")

# Faire une prédiction
prediction = best_model.predict(example_patient)[0]
prediction_proba = best_model.predict_proba(example_patient)[0]

print(f"\nPrédiction du modèle ({best_model_name}) :")
print(f"Résultat : {'Malade' if prediction == 1 else 'Non malade'}")
print(f"Probabilités :")
print(f"  - Non malade : {prediction_proba[0]:.3f} ({prediction_proba[0]*100:.1f}%)")
print(f"  - Malade : {prediction_proba[1]:.3f} ({prediction_proba[1]*100:.1f}%)")

# Vérifier si la prédiction est correcte
if prediction == true_label:
    print("\nPrédiction CORRECTE")
else:
    print("\nPrédiction INCORRECTE")

## Résumé

### Ce que nous avons accompli :

1. **Préparation des données** :
   - Remplissage des valeurs manquantes
   - Encodage des variables catégorielles
   - Séparation variables/cible
   - Division entraînement/test (80%/20%)
   - Standardisation des variables

2. **Modèles entraînés** :
   - Régression Logistique
   - Forêt Aléatoire
   - SVM

3. **Évaluation des performances** :
   - Accuracy et AUC pour chaque modèle
   - Matrices de confusion
   - Rapports de classification détaillés

4. **Meilleur modèle identifié** :
   - Basé sur les métriques combinées
   - Test sur un exemple concret

### Interprétation médicale :

- **Accuracy** : Pourcentage de prédictions correctes globales
- **Precision** : Fiabilité des prédictions positives
- **Recall** : Capacité à détecter tous les malades
- **F1-score** : Équilibre entre precision et recall

### Prochaines étapes possibles :
1. Optimiser les hyperparamètres
2. Essayer des modèles plus avancés
3. Analyser l'importance des variables
4. Sauvegarder le modèle pour déploiement

**Félicitations ! Vous avez créé votre premier pipeline de machine learning !**